<!-- notebook-header -->
# Segmentacao de Imagens

**Modulo:** 05 - Dominios Aplicados / 05A - Computer Vision  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Segmentacao semantica, instance segmentation, panoptic segmentation, U-Net e metricas.


# Segmentacao de Imagens

## Objetivo
Entender como classificar cada pixel de uma imagem em uma categoria, desde semantic segmentation
(classificar todos os pixels) ate instance segmentation (distinguir objetos individuais).

## Pre-requisitos
- 5A_1 (CNN Fundamentos): convolucao, pooling, feature maps
- 5A_3 (Deteccao): bounding boxes, IoU, Mask R-CNN

## Conteudo
1. Tipos de Segmentacao
2. Encoder-Decoder e U-Net
3. Atrous Convolutions e DeepLab
4. Instance e Panoptic Segmentation
5. Loss Functions para Segmentacao
6. Metricas (IoU, Dice)
7. Exercicios Praticos
8. Erros Comuns e Armadilhas
9. Resumo e Conexoes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
np.random.seed(42)
print('Imports OK')

## 1. Tipos de Segmentacao

### Analogia: Colorir um Desenho

Imagine que voce tem uma foto e precisa "colorir" cada pixel:
- **Semantic Segmentation:** pinte todos os gatos de vermelho, todos os cachorros de azul.
  Nao importa se ha 2 gatos -- ambos ficam vermelhos.
- **Instance Segmentation:** pinte gato 1 de vermelho e gato 2 de laranja.
  Cada objeto individual tem sua propria cor.
- **Panoptic Segmentation:** combine ambos -- gatos individuais + ceu + chao como categorias.

### Definicao Formal

| Tarefa | Classifica Pixels | Distingue Instancias | Exemplo |
|--------|-------------------|---------------------|---------|
| Semantic | Sim (todas as classes) | Nao | Todos os carros = mesma cor |
| Instance | Sim (objetos "things") | Sim | Carro 1 vs Carro 2 |
| Panoptic | Sim (things + stuff) | Sim (so things) | Carros individuais + ceu + estrada |

### Por que em ML: Segmentacao como Visao "Densa"

Classificacao diz "ha um gato"; deteccao diz "o gato esta aqui (box)"; segmentacao diz
"estes exatos pixels sao o gato". E a forma mais completa de entender uma imagem,
essencial para carros autonomos, medicina, e edicao de imagens.

In [ ]:

# Criar exemplo visual
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Original image (dummy)
original = np.random.rand(100, 100, 3)
axes[0].imshow(original)
axes[0].set_title('Original Image')
axes[0].axis('off')

# Semantic segmentation
semantic = np.zeros((100, 100))
semantic[20:40, 20:60] = 1  # Cat 1
semantic[60:80, 30:70] = 1  # Cat 2 (same class)
axes[1].imshow(semantic, cmap='viridis')
axes[1].set_title('Semantic Segmentation\n(Gatos = Mesma cor)')
axes[1].axis('off')

# Instance segmentation
instance = np.zeros((100, 100))
instance[20:40, 20:60] = 1  # Cat instance 1
instance[60:80, 30:70] = 2  # Cat instance 2 (different ID)
axes[2].imshow(instance, cmap='tab20')
axes[2].set_title('Instance Segmentation\n(Gatos = Cores diferentes)')
axes[2].axis('off')

# Panoptic segmentation
panoptic = np.zeros((100, 100))
panoptic[:20, :] = 10  # Sky (stuff)
panoptic[20:40, 20:60] = 1  # Cat instance 1 (thing)
panoptic[60:80, 30:70] = 2  # Cat instance 2 (thing)
axes[3].imshow(panoptic, cmap='tab20')
axes[3].set_title('Panoptic Segmentation\n(Stuff + Things)')
axes[3].axis('off')

plt.tight_layout()
plt.savefig('/tmp/segmentation_types.png', dpi=100, bbox_inches='tight')
plt.show()


### O que observar sobre a Escolha do Tipo de Segmentacao

A escolha depende da aplicacao:
- **Autonomous driving:** panoptic (precisa distinguir carros individuais E segmentar estrada)
- **Imagens medicas:** semantic (tumor vs nao-tumor, geralmente 1 instancia por imagem)
- **Edicao de fotos:** instance (separar pessoa do fundo para trocar background)
- **Satelite:** semantic (uso do solo: floresta, agua, urbano)

### O que concluir sobre Complexidade Crescente

Semantic < Instance < Panoptic em complexidade. Semantic e um problema de classificacao
por pixel; Instance adiciona deteccao; Panoptic unifica tudo. Na pratica, comece com
semantic (mais simples) e adicione complexidade conforme necessario.

### Conexao com outros notebooks sobre Deteccao

Instance segmentation estende diretamente 5A_3 (deteccao). Mask R-CNN = Faster R-CNN +
branch de mascara. Se voce ja tem um detector funcionando, adicionar segmentacao e
relativamente simples (1 branch FCN a mais).

## 2. Encoder-Decoder e U-Net

### Analogia: Comprimir e Descomprimir uma Foto

O encoder e como comprimir uma foto JPEG: voce perde detalhes mas captura a essencia.
O decoder e como descomprimir: tenta reconstruir os detalhes. **Skip connections** sao como
guardar "notas" dos detalhes perdidos para ajudar na reconstrucao.

Sem skip connections: a foto reconstruida fica borrada.
Com skip connections: os detalhes finos (bordas, texturas) sao preservados.

### Definicao Formal: U-Net

U-Net (2015, Ronneberger) tem formato de U:
- **Encoder (esquerda):** sequencia de Conv + Pool que reduz resolucao (256->128->64->32->16)
- **Bottleneck (fundo):** feature map de menor resolucao com maior semantica
- **Decoder (direita):** sequencia de Upsample + Conv que recupera resolucao
- **Skip connections:** concatenam features do encoder com o decoder correspondente

Cada nivel do decoder recebe: features upsampled do nivel anterior + features do encoder
(via skip connection). Isso combina semantica (do fundo) com detalhes (do encoder).

### Por que em ML: U-Net como Arquitetura Dominante

U-Net e a arquitetura mais citada em segmentacao medica. Funciona bem com datasets
pequenos (< 100 imagens) gracas aos skip connections que regularizam o treino.
Variantes modernas (U-Net++, Attention U-Net, nnU-Net) dominam benchmarks medicos.

In [ ]:
# Visualizacao: Arquitetura U-Net
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('U-Net: Encoder-Decoder com Skip Connections', fontsize=16, fontweight='bold')

# Encoder blocks (left side)
encoder_blocks = [
    (1, 7, 2, 2, '3->64\n256x256', '#4CAF50'),    # enc1
    (1, 4.5, 1.8, 1.8, '64->128\n128x128', '#2196F3'), # enc2
    (1, 2.2, 1.5, 1.5, '128->256\n64x64', '#FF9800'),   # enc3
    (1, 0.3, 1.2, 1.2, '256->512\n32x32', '#F44336'),   # enc4
]

# Bottleneck
bottleneck = (4.5, 0.3, 1, 1, '512->1024\n16x16', '#9C27B0')

# Decoder blocks (right side)
decoder_blocks = [
    (8, 0.3, 1.2, 1.2, '1024->512\n32x32', '#F44336'),
    (8.5, 2.2, 1.5, 1.5, '512->256\n64x64', '#FF9800'),
    (9, 4.5, 1.8, 1.8, '256->128\n128x128', '#2196F3'),
    (9.5, 7, 2, 2, '128->64\n256x256', '#4CAF50'),
]

# Draw encoder
for x, y, w, h, label, color in encoder_blocks:
    rect = patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                   facecolor=color, alpha=0.6, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=8, fontweight='bold')

# Draw bottleneck
x, y, w, h, label, color = bottleneck
rect = patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                               facecolor=color, alpha=0.6, edgecolor='black')
ax.add_patch(rect)
ax.text(x + w/2, y + h/2, label, ha='center', va='center',
        fontsize=8, fontweight='bold', color='white')

# Draw decoder
for x, y, w, h, label, color in decoder_blocks:
    rect = patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                   facecolor=color, alpha=0.3, edgecolor='black', linestyle='--')
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=8, fontweight='bold')

# Downsampling arrows (encoder)
for i in range(len(encoder_blocks) - 1):
    x1 = encoder_blocks[i][0] + encoder_blocks[i][2] / 2
    y1 = encoder_blocks[i][1]
    y2 = encoder_blocks[i+1][1] + encoder_blocks[i+1][3]
    ax.annotate('', xy=(x1, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    ax.text(x1 + 0.3, (y1 + y2)/2, 'Pool', fontsize=7, color='gray')

# Encoder to bottleneck
ax.annotate('', xy=(4.5, 0.8), xytext=(2.2, 0.8),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# Bottleneck to decoder
ax.annotate('', xy=(8, 0.8), xytext=(5.5, 0.8),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# Upsampling arrows (decoder)
for i in range(len(decoder_blocks) - 1):
    x1 = decoder_blocks[i][0] + decoder_blocks[i][2] / 2
    y1 = decoder_blocks[i][1] + decoder_blocks[i][3]
    y2 = decoder_blocks[i+1][1]
    ax.annotate('', xy=(x1 + 0.5, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='purple', lw=2))
    ax.text(x1 + 0.5, (y1 + y2)/2, 'Up', fontsize=7, color='purple')

# Skip connections (curved arrows)
skip_colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
for i in range(len(encoder_blocks)):
    ex = encoder_blocks[i][0] + encoder_blocks[i][2]
    ey = encoder_blocks[i][1] + encoder_blocks[i][3] / 2
    dx = decoder_blocks[3-i][0]
    dy = decoder_blocks[3-i][1] + decoder_blocks[3-i][3] / 2
    ax.annotate('', xy=(dx, dy), xytext=(ex, ey),
                arrowprops=dict(arrowstyle='->', color=skip_colors[i],
                               lw=2, linestyle='--',
                               connectionstyle='arc3,rad=-0.3'))

# Labels
ax.text(2, 9.5, 'ENCODER\n(Downsampling)', ha='center', fontsize=12, fontweight='bold', color='green')
ax.text(5, 9.5, 'BOTTLENECK', ha='center', fontsize=12, fontweight='bold', color='purple')
ax.text(10, 9.5, 'DECODER\n(Upsampling)', ha='center', fontsize=12, fontweight='bold', color='orange')
ax.text(14, 5, 'Skip\nConnections\n(concatenar)', fontsize=10, color='gray',
        ha='center', style='italic')

# Output
ax.text(12, 8, 'Output:\n64->num_classes\n(Conv 1x1)', fontsize=9,
        ha='center', bbox=dict(boxstyle='round', facecolor='gold', alpha=0.5))

plt.tight_layout()
plt.savefig('/tmp/unet_architecture.png', dpi=100, bbox_inches='tight')
plt.show()

print('U-Net: skip connections preservam detalhes espaciais')
print('Sem skip connections: segmentacao borrada nas bordas')
print('Com skip connections: bordas nitidas e detalhes preservados')

In [ ]:
# Demonstracao: efeito de skip connections na segmentacao
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

np.random.seed(42)

# Simular uma imagem com objetos definidos
img = np.zeros((64, 64))
# Objeto 1: circulo
for i in range(64):
    for j in range(64):
        if (i - 20)**2 + (j - 20)**2 < 100:
            img[i, j] = 1.0
# Objeto 2: retangulo
img[35:55, 35:55] = 2.0
# Bordas finas
img[10, 10:50] = 3.0

# Ground truth
gt = img.copy()

# Manual gaussian blur (sem scipy)
def manual_blur(arr, sigma_equiv=3):
    """Blur simples usando media em janela."""
    k = max(1, int(sigma_equiv))
    result = np.zeros_like(arr, dtype=float)
    h, w = arr.shape
    for i in range(h):
        for j in range(w):
            i_start = max(0, i - k)
            i_end = min(h, i + k + 1)
            j_start = max(0, j - k)
            j_end = min(w, j + k + 1)
            result[i, j] = arr[i_start:i_end, j_start:j_end].mean()
    return result

# Predicao SEM skip connections (borrada)
pred_no_skip_raw = manual_blur(img.astype(float), sigma_equiv=3)
pred_no_skip = np.round(pred_no_skip_raw).astype(int)
pred_no_skip = np.clip(pred_no_skip, 0, 3)

# Predicao COM skip connections (nitida -- pouco blur)
pred_with_skip_raw = manual_blur(img.astype(float), sigma_equiv=1)
pred_with_skip = np.round(pred_with_skip_raw).astype(int)
pred_with_skip = np.clip(pred_with_skip, 0, 3)

# Row 1: Predictions
axes[0,0].imshow(gt, cmap='tab10', vmin=0, vmax=3)
axes[0,0].set_title('Ground Truth', fontsize=12, fontweight='bold')
axes[0,0].axis('off')

axes[0,1].imshow(pred_no_skip, cmap='tab10', vmin=0, vmax=3)
axes[0,1].set_title('Sem Skip Connections\n(bordas borradas)', fontsize=12, fontweight='bold')
axes[0,1].axis('off')

axes[0,2].imshow(pred_with_skip, cmap='tab10', vmin=0, vmax=3)
axes[0,2].set_title('Com Skip Connections\n(bordas nitidas)', fontsize=12, fontweight='bold')
axes[0,2].axis('off')

# Row 2: Error maps
axes[1,0].set_visible(False)

error_no_skip = (pred_no_skip != gt).astype(float)
axes[1,1].imshow(error_no_skip, cmap='Reds')
n_errors_no = error_no_skip.sum()
axes[1,1].set_title(f'Erros: {int(n_errors_no)} pixels', fontsize=12, fontweight='bold', color='red')
axes[1,1].axis('off')

error_with_skip = (pred_with_skip != gt).astype(float)
axes[1,2].imshow(error_with_skip, cmap='Reds')
n_errors_with = error_with_skip.sum()
axes[1,2].set_title(f'Erros: {int(n_errors_with)} pixels', fontsize=12, fontweight='bold', color='red')
axes[1,2].axis('off')

plt.suptitle('Impacto dos Skip Connections na Qualidade da Segmentacao', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/skip_connections_effect.png', dpi=100, bbox_inches='tight')
plt.show()

if n_errors_no > 0:
    improvement = (n_errors_no - n_errors_with) / n_errors_no * 100
    print(f'Skip connections reduziram erros em {improvement:.0f}%')
print('A maioria dos erros SEM skip connections esta nas BORDAS dos objetos')

### O que observar sobre Variantes de U-Net

| Variante | Inovacao | Quando usar |
|----------|----------|-------------|
| U-Net | Skip connections por concatenacao | Baseline, datasets pequenos |
| U-Net++ | Dense skip connections (nested) | Quando precisao maxima importa |
| Attention U-Net | Attention gates nos skip connections | Objetos de tamanho variavel |
| nnU-Net | Auto-configuracao de hiperparametros | Qualquer tarefa medica (SOTA) |
| V-Net | 3D U-Net com Dice Loss | Volumes 3D (CT, MRI) |

### O que concluir sobre Transfer Learning em Segmentacao

Assim como em classificacao, o encoder da U-Net pode ser inicializado com pesos
pre-treinados (ImageNet). Isso e especialmente impactante com datasets pequenos:
encoder pre-treinado + decoder treinado do zero pode dar +10% mIoU.

### Conexao com outros notebooks sobre Arquiteturas CNN

U-Net usa exatamente os building blocks de 5A_1: convolucao, pooling, batch normalization.
O encoder e essencialmente uma CNN de classificacao (ResNet, EfficientNet) sem as camadas
finais. A inovacao esta no decoder com skip connections.

## 3. Atrous Convolutions e DeepLab

### Analogia: Olhar pela Janela com Diferentes Angulos

Uma convolucao normal 3x3 olha para 9 pixels adjacentes. E como olhar por uma janela
pequena. **Atrous (dilated) convolution** e como olhar pela mesma janela, mas com
buracos: voce ve uma area maior usando os mesmos 9 parametros.

- Dilation 1 (normal): ve 3x3 = area de 9 pixels
- Dilation 2: ve 5x5 = area de 25 pixels (mas so usa 9 parametros)
- Dilation 4: ve 9x9 = area de 81 pixels (mas so usa 9 parametros)

Resultado: contexto maior sem mais parametros e sem perder resolucao!

### Definicao Formal: ASPP

**Atrous Spatial Pyramid Pooling (ASPP)** usa multiplas dilated convolutions em paralelo:
- Conv 1x1 (contexto local)
- Atrous 3x3, dilation=6 (contexto medio)
- Atrous 3x3, dilation=12 (contexto grande)
- Atrous 3x3, dilation=18 (contexto muito grande)
- Global Average Pooling (contexto global)

Concatenar todas as saidas captura informacao em TODAS as escalas simultaneamente.

### Por que em ML: DeepLab como SOTA em Semantic Segmentation

DeepLabv3+ combina ASPP (multi-scale) com encoder-decoder (detalhes espaciais).
E consistentemente top-3 em benchmarks como Cityscapes, PASCAL VOC, e ADE20K.

In [ ]:

# Visualizar Atrous Convolution
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

# Regular convolution (dilation=1)
ax = axes[0, 0]
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 5.5)
ax.set_aspect('equal')
ax.set_title('Regular Conv (dilation=1, kernel=3x3)')

# Desenhar grid
for i in range(6):
    ax.axhline(y=i-0.5, color='gray', linewidth=0.5, alpha=0.3)
    ax.axvline(x=i-0.5, color='gray', linewidth=0.5, alpha=0.3)

# Kernel positions
kernel_pos_regular = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)]
for pos in kernel_pos_regular:
    circle = plt.Circle(pos, 0.15, color='red', alpha=0.7)
    ax.add_patch(circle)

# Center
center_circle = plt.Circle((2, 2), 0.2, color='blue', alpha=0.9)
ax.add_patch(center_circle)

ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.grid(True, alpha=0.3)

# Atrous convolution (dilation=2)
ax = axes[0, 1]
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 5.5)
ax.set_aspect('equal')
ax.set_title('Atrous Conv (dilation=2, kernel=3x3)')

for i in range(6):
    ax.axhline(y=i-0.5, color='gray', linewidth=0.5, alpha=0.3)
    ax.axvline(x=i-0.5, color='gray', linewidth=0.5, alpha=0.3)

kernel_pos_dilated = [(1, 1), (1, 3), (3, 1), (3, 3), (2, 2)]
for pos in kernel_pos_dilated:
    circle = plt.Circle(pos, 0.15, color='green', alpha=0.7)
    ax.add_patch(circle)

center_circle = plt.Circle((2, 2), 0.2, color='blue', alpha=0.9)
ax.add_patch(center_circle)

ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.grid(True, alpha=0.3)

# Atrous convolution (dilation=3)
ax = axes[1, 0]
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 5.5)
ax.set_aspect('equal')
ax.set_title('Atrous Conv (dilation=3, kernel=3x3)')

for i in range(6):
    ax.axhline(y=i-0.5, color='gray', linewidth=0.5, alpha=0.3)
    ax.axvline(x=i-0.5, color='gray', linewidth=0.5, alpha=0.3)

ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.grid(True, alpha=0.3)

# Receptive Field Comparison
ax = axes[1, 1]
ax.axis('off')

rf_text = '''Receptive Field Comparison:

Regular Conv (3x3, dilation=1):
  - Receptive Field: 3x3
  - Parameters: Same

Atrous Conv (3x3, dilation=2):
  - Receptive Field: 5x5
  - Parameters: Same
  - Vantagem: Maior contexto sem perder resolução

Atrous Conv (3x3, dilation=3):
  - Receptive Field: 7x7
  - Parameters: Same
  - Uso: Camadas profundas para contexto global

ASPP (Atrous Spatial Pyramid Pooling):
  - Múltiplas dilations em paralelo: 1, 6, 12, 18
  - Capture features em múltiplas escalas
  - Concatenar todas as saídas
  - Refinar com 1x1 convolution
'''

ax.text(0.05, 0.95, rf_text, transform=ax.transAxes,
       fontsize=9, verticalalignment='top', family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.savefig('/tmp/atrous_convolution.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Comparacao: U-Net vs DeepLab
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Architecture comparison diagram
models = {
    'U-Net': {
        'approach': 'Encoder-Decoder\n+ Skip Connections',
        'context': 'Hierarquico\n(multi-level)',
        'detail': 'Skip connections\n(concatenacao)',
        'best_for': 'Datasets pequenos\nImagens medicas',
        'miou': 68,
        'params': '31M',
    },
    'DeepLabv3+': {
        'approach': 'ASPP\n+ Encoder-Decoder',
        'context': 'Multi-scale\n(atrous parallel)',
        'detail': 'Decoder leve\n+ low-level features',
        'best_for': 'Datasets grandes\nNatural scenes',
        'miou': 79,
        'params': '41M',
    },
}

ax = axes[0]
ax.axis('off')
ax.set_title('U-Net vs DeepLabv3+', fontsize=14, fontweight='bold')

y_pos = 0.9
for name, info in models.items():
    ax.text(0.05, y_pos, name, fontsize=14, fontweight='bold',
            transform=ax.transAxes, color='navy')
    y_pos -= 0.06
    for key, val in info.items():
        if key != 'miou' and key != 'params':
            ax.text(0.08, y_pos, f'{key}: {val}', fontsize=9,
                    transform=ax.transAxes)
            y_pos -= 0.05
    y_pos -= 0.08

# Bar chart: model comparison
ax = axes[1]
model_names = ['FCN-8s', 'U-Net', 'PSPNet', 'DeepLabv3', 'DeepLabv3+', 'SegFormer']
miou_vals = [62.2, 67.8, 73.2, 77.1, 79.3, 82.4]
years = [2015, 2015, 2017, 2017, 2018, 2021]
colors = ['gray', '#4CAF50', 'gray', '#FF9800', '#F44336', '#2196F3']

bars = ax.barh(range(len(model_names)), miou_vals, color=colors, alpha=0.7)
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels([f'{n} ({y})' for n, y in zip(model_names, years)])
ax.set_xlabel('mIoU (Cityscapes val)', fontsize=12)
ax.set_title('Evolucao da Semantic Segmentation', fontsize=13, fontweight='bold')

for bar, val in zip(bars, miou_vals):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(55, 90)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/tmp/segmentation_models.png', dpi=100, bbox_inches='tight')
plt.show()

print('Tendencia: SegFormer (Transformer) superou DeepLab (CNN)')
print('Porem, U-Net continua SOTA em imagens medicas com datasets pequenos')

### O que observar sobre o Trade-off Resolucao vs Contexto

O dilema central de segmentacao: pooling aumenta contexto mas perde resolucao.
Solucoes:
- **U-Net:** pool + upsample com skip connections
- **DeepLab:** atrous convolutions (contexto sem perder resolucao)
- **PSPNet:** pyramid pooling module
- **SegFormer:** self-attention (contexto global sem pooling)

### O que concluir sobre Quando Usar Qual Arquitetura

Regra pratica:
- Dataset < 1K imagens: U-Net com encoder pre-treinado
- Dataset > 10K, scenes naturais: DeepLabv3+ ou SegFormer
- Volume 3D (medico): nnU-Net (auto-configuracao)
- Real-time: BiSeNet ou STDC

### Conexao com outros notebooks sobre Receptive Field

Atrous convolutions conectam com 5A_1 (receptive field em CNNs). La vimos que
receptive field cresce com profundidade; aqui, atrous convolutions oferecem um
atalho para aumentar receptive field sem profundidade adicional.

## 4. Instance e Panoptic Segmentation

### Analogia: De Pintura a Escultura

Semantic segmentation e como uma pintura plana: tudo em 2D, sem profundidade.
Instance segmentation e como fazer esculturas individuais de cada objeto:
cada gato e uma escultura separada, com sua propria forma. Panoptic e
construir um diorama completo: esculturas individuais + cenario pintado.

### Definicao Formal: Mask R-CNN

Mask R-CNN = Faster R-CNN + branch de mascara:
1. **Backbone + FPN:** extrair features multi-escala
2. **RPN:** gerar region proposals
3. **RoI Align:** extrair features para cada proposta (sem quantizacao!)
4. **Heads paralelas:** classificacao + bbox regression + **mascara 28x28**

A inovacao chave: **RoI Align** usa interpolacao bilinear em vez de quantizacao,
eliminando desalinhamento espacial que degradava mascaras.

### Por que em ML: Panoptic como Visao Completa

Panoptic Segmentation (Kirillov 2019) unifica semantic + instance:
- "Things" (objetos contaveis): instance segmentation (carro 1, carro 2)
- "Stuff" (nao-contavel): semantic segmentation (ceu, estrada, grama)

Modelos modernos como Mask2Former resolvem ambos com uma unica arquitetura.

In [ ]:

# Exemplo conceitual de Instance Segmentation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Imagem original
ax = axes[0]
original = np.random.rand(100, 100, 3)
ax.imshow(original)
ax.set_title('Original Image')
ax.axis('off')

# Detecções com bboxes
ax = axes[1]
ax.imshow(original)
# Bbox 1 (gato)
rect1 = patches.Rectangle((20, 20), 40, 40, linewidth=2, edgecolor='red', facecolor='none')
ax.add_patch(rect1)
ax.text(20, 15, 'Cat', color='red', fontsize=10, weight='bold')
# Bbox 2 (cachorro)
rect2 = patches.Rectangle((60, 50), 30, 35, linewidth=2, edgecolor='blue', facecolor='none')
ax.add_patch(rect2)
ax.text(60, 45, 'Dog', color='blue', fontsize=10, weight='bold')
ax.set_title('Bounding Boxes (Faster R-CNN)')
ax.axis('off')

# Instance segmentation masks
ax = axes[2]
masks = np.zeros((100, 100, 3))
# Cat mask
cat_mask = np.zeros((100, 100))
cat_mask[25:60, 25:60] = 1
# Dog mask
dog_mask = np.zeros((100, 100))
dog_mask[55:90, 60:90] = 1
# Visualize
masks[:, :, 0] = cat_mask * 0.7  # Red
masks[:, :, 2] = dog_mask * 0.7  # Blue
ax.imshow(masks)
ax.set_title('Instance Masks (Mask R-CNN)')
ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/instance_segmentation.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Comparacao: Semantic vs Instance vs Panoptic
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

np.random.seed(42)

# Criar cena simulada
scene = np.zeros((100, 100))
# Sky (stuff)
scene[:30, :] = 1
# Building (stuff)
scene[30:70, 10:40] = 2
scene[30:70, 60:90] = 2
# Road (stuff)
scene[70:, :] = 3
# Car 1 (thing)
scene[75:90, 15:35] = 4
# Car 2 (thing)
scene[75:90, 55:75] = 4
# Person 1 (thing)
scene[55:70, 45:52] = 5
# Person 2 (thing)
scene[60:70, 78:85] = 5

# Original
axes[0,0].imshow(scene, cmap='tab10')
axes[0,0].set_title('Cena Original', fontsize=12, fontweight='bold')
axes[0,0].axis('off')

# Semantic
semantic = scene.copy()
# carros = mesma classe, pessoas = mesma classe
axes[0,1].imshow(semantic, cmap='tab10')
axes[0,1].set_title('Semantic Segmentation\n(mesma cor por classe)', fontsize=12, fontweight='bold')
axes[0,1].axis('off')
# Add legend
for cls, name, color in [(1,'ceu','tab:blue'), (2,'predio','tab:orange'),
                          (3,'rua','tab:green'), (4,'carro','tab:red'), (5,'pessoa','tab:purple')]:
    axes[0,1].plot([], [], 's', color=color, label=name, markersize=10)
axes[0,1].legend(loc='lower left', fontsize=8)

# Instance
instance = np.zeros((100, 100))
instance[:30, :] = 1    # sky
instance[30:70, 10:40] = 2  # building
instance[30:70, 60:90] = 2  # building
instance[70:, :] = 3    # road
instance[75:90, 15:35] = 4  # car 1
instance[75:90, 55:75] = 5  # car 2 (different!)
instance[55:70, 45:52] = 6  # person 1
instance[60:70, 78:85] = 7  # person 2 (different!)
axes[1,0].imshow(instance, cmap='tab20')
axes[1,0].set_title('Instance Segmentation\n(cor diferente por instancia)', fontsize=12, fontweight='bold')
axes[1,0].axis('off')

# Panoptic (highlight things vs stuff)
panoptic = instance.copy()
axes[1,1].imshow(panoptic, cmap='tab20')
axes[1,1].set_title('Panoptic Segmentation\n(stuff + things)', fontsize=12, fontweight='bold')
axes[1,1].axis('off')

# Annotate stuff vs things
axes[1,1].text(50, 15, 'STUFF', ha='center', fontsize=11, color='white', fontweight='bold')
axes[1,1].text(50, 50, 'STUFF', ha='center', fontsize=11, color='white', fontweight='bold')
axes[1,1].text(25, 82, 'THING 1', ha='center', fontsize=8, color='white', fontweight='bold')
axes[1,1].text(65, 82, 'THING 2', ha='center', fontsize=8, color='white', fontweight='bold')

plt.suptitle('Tipos de Segmentacao: Mesma Cena, Outputs Diferentes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/seg_types_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('Semantic: 5 classes (ceu, predio, rua, carro, pessoa)')
print('Instance: 7 IDs (2 carros distintos, 2 pessoas distintas)')
print('Panoptic: combina ambos (stuff nao tem instancias)')

### O que observar sobre SAM (Segment Anything Model)

SAM (Meta, 2023) mudou o paradigma: um unico modelo que segmenta QUALQUER objeto
em QUALQUER imagem, sem treino especifico. Funciona com prompts:
- Ponto: "segmente o objeto neste pixel"
- Box: "segmente dentro desta caixa"
- Texto (SAM 2): "segmente todos os gatos"

SAM foi treinado em 11M imagens com 1B mascaras, criando um "foundation model" para segmentacao.

### O que concluir sobre a Unificacao de Tarefas

A tendencia moderna e unificar deteccao + segmentacao em um framework:
- **Mask2Former:** semantic + instance + panoptic com mesma arquitetura
- **DINO + SAM:** deteccao open-vocabulary + segmentacao zero-shot
- **OneFormer:** modelo unico para todas as tarefas de segmentacao

### Conexao com outros notebooks sobre Mask R-CNN

Mask R-CNN estende diretamente Faster R-CNN (5A_3). A unica adicao e uma branch FCN
que prediz mascara 28x28 para cada RoI. RoI Align (vs RoI Pooling) e essencial
para preservar alinhamento espacial nas mascaras.

## 5. Loss Functions e Metricas

### Analogia: Corrigindo uma Prova

Imagine corrigir uma prova de colorir:
- **Pixel-wise Cross-Entropy:** cada pixel errado vale -1 ponto (igual peso)
- **Dice Loss:** mede a "sobreposicao" entre resposta e gabarito (melhor para formas)
- **Focal Loss:** erros em pixels "faceis" (fundo) valem menos que erros em pixels "dificeis"

### Definicao Formal

**Cross-Entropy por pixel:**
L_CE = -1/N * sum(y_i * log(p_i)) para cada pixel i

**Dice Loss:**
L_Dice = 1 - (2 * |A inter B| + smooth) / (|A| + |B| + smooth)

**IoU (Jaccard Index):**
IoU = |A inter B| / |A union B|

**Relacao:** Dice = 2*IoU / (1+IoU). Dice >= IoU sempre.

### Por que em ML: Dice Loss como Default para Segmentacao

Cross-Entropy trata cada pixel independentemente; Dice Loss avalia a *forma* como um todo.
Para segmentacao medica (onde o tumor e 1% da imagem), Dice Loss e muito superior
porque nao e dominada pelos pixels de background.

In [ ]:
# Demonstracao: Dice Score e IoU
def compute_dice_score(pred, gt):
    """Dice Score entre duas mascaras binarias."""
    intersection = np.logical_and(pred, gt).sum()
    return (2 * intersection) / (pred.sum() + gt.sum() + 1e-7)

def compute_iou_score(pred, gt):
    """IoU entre duas mascaras binarias."""
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return intersection / (union + 1e-7)

# Cenarios de comparacao
np.random.seed(42)
gt_mask = np.zeros((50, 50), dtype=bool)
gt_mask[15:35, 15:35] = True  # quadrado 20x20

scenarios = {
    'Perfeito': gt_mask.copy(),
    'Bom (offset)': np.zeros((50, 50), dtype=bool),
    'Medio (menor)': np.zeros((50, 50), dtype=bool),
    'Ruim (grande)': np.zeros((50, 50), dtype=bool),
    'Pessimo': np.zeros((50, 50), dtype=bool),
    'Vazio': np.zeros((50, 50), dtype=bool),
}
scenarios['Bom (offset)'][17:37, 17:37] = True
scenarios['Medio (menor)'][20:30, 20:30] = True
scenarios['Ruim (grande)'][5:45, 5:45] = True
scenarios['Pessimo'][0:10, 0:10] = True

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (name, pred) in enumerate(scenarios.items()):
    ax = axes[i]
    # Overlap visualization
    overlap = np.zeros((50, 50, 3))
    overlap[:,:,1] = gt_mask * 0.5     # GT = green
    overlap[:,:,0] = pred * 0.5         # Pred = red
    overlap[:,:,1] += np.logical_and(pred, gt_mask) * 0.5  # overlap = yellow-ish

    ax.imshow(overlap)

    dice = compute_dice_score(pred, gt_mask)
    iou = compute_iou_score(pred, gt_mask)
    ax.set_title(f'{name}\nDice={dice:.3f}  IoU={iou:.3f}', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Dice Score vs IoU para Diferentes Predicoes\n(Verde=GT, Vermelho=Pred, Amarelo=Overlap)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/dice_vs_iou.png', dpi=100, bbox_inches='tight')
plt.show()

print('Observacoes:')
print('  - Dice >= IoU sempre (Dice e mais "generoso")')
print('  - Offset pequeno: IoU cai mais rapido que Dice')
print('  - Predicao vazia: ambos = 0')

In [ ]:
# Demonstracao: CE vs Dice Loss em cenario desbalanceado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simular cenario desbalanceado: 95% background, 5% foreground
np.random.seed(42)
sizes = np.arange(0.01, 0.51, 0.01)  # foreground ratio

# Cross-entropy: predizer tudo como background
ce_loss_all_bg = []
ce_loss_perfect = []
dice_loss_all_bg = []
dice_loss_perfect = []

for fg_ratio in sizes:
    n = 10000
    n_fg = int(n * fg_ratio)
    n_bg = n - n_fg

    # All-background prediction
    ce_bg = -np.log(1 - fg_ratio + 1e-7)  # loss for fg pixels predicted as bg
    ce_loss_all_bg.append(fg_ratio * ce_bg)

    # Perfect prediction
    ce_loss_perfect.append(0.001)

    # Dice: all background
    pred_all_bg = np.zeros(n)
    gt = np.zeros(n)
    gt[:n_fg] = 1
    dice_all_bg = 1 - compute_dice_score(pred_all_bg > 0.5, gt > 0.5)
    dice_loss_all_bg.append(dice_all_bg)

    # Dice: perfect
    dice_loss_perfect.append(0.001)

ax = axes[0]
ax.plot(sizes * 100, ce_loss_all_bg, 'r-', linewidth=2, label='CE: pred=all BG')
ax.axhline(y=0.05, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('% Foreground', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Cross-Entropy Loss\n(predizer tudo como background)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.annotate('Com 5% FG, CE loss e BAIXO\nmesmo predizendo tudo errado!',
           xy=(5, ce_loss_all_bg[4]), xytext=(20, 0.15),
           arrowprops=dict(arrowstyle='->', color='red'),
           fontsize=10, color='red')

ax = axes[1]
ax.plot(sizes * 100, dice_loss_all_bg, 'b-', linewidth=2, label='Dice: pred=all BG')
ax.axhline(y=0.05, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('% Foreground', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Dice Loss\n(predizer tudo como background)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.annotate('Com 5% FG, Dice loss e ALTO\npenaliza fortemente!',
           xy=(5, dice_loss_all_bg[4]), xytext=(20, 0.5),
           arrowprops=dict(arrowstyle='->', color='blue'),
           fontsize=10, color='blue')

plt.suptitle('Por que Dice Loss e Melhor para Dados Desbalanceados', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/ce_vs_dice_imbalance.png', dpi=100, bbox_inches='tight')
plt.show()

print('CE Loss: com 95% background, predizer tudo como BG tem loss baixo!')
print('Dice Loss: penaliza forte mesmo quando FG e pequeno')
print('Por isso Dice e o default para segmentacao medica')

### O que observar sobre Combinacao de Losses

Na pratica, a melhor estrategia e combinar losses:
- **CE + Dice:** CE treina rapido no inicio, Dice refina as formas no final
- **Focal + Dice:** para cenarios com imbalance extremo
- **Boundary Loss:** adiciona penalidade extra nas bordas dos objetos

Formula tipica: L_total = alpha * L_CE + beta * L_Dice + gamma * L_Boundary

### O que concluir sobre mIoU como Metrica Principal

mIoU (mean Intersection over Union) e a metrica padrao para segmentacao.
E calculado por classe e depois promediado. Cuidado: classes raras tem peso
igual a classes comuns na media, o que pode mascarar performance ruim em classes raras.
Sempre reporte IoU por classe alem de mIoU.

### Conexao com outros notebooks sobre Metricas

Dice e IoU conectam com 2_4 (metricas) e 5A_3 (IoU para deteccao). A diferenca e que
em segmentacao, IoU e calculado pixel-a-pixel, nao box-a-box. A curva PR de 5A_3
tem analogo em segmentacao: variar o threshold de confidence e plotar precision vs recall.

## 7. Exercicios Praticos

### Exercicio 1: Implementar Dice Score Multi-Classe
Calcule Dice Score por classe e mDice para uma predicao multi-classe.

Faca o seguinte:
- Implemente `multi_class_dice(pred, gt, num_classes)`
- Retorne Dice por classe e media
- Teste com predicao simulada

In [ ]:
# PRATICA - Exercicio 1: Dice Score Multi-Classe
def multi_class_dice(pred, gt, num_classes):
    """
    Calcular Dice Score por classe.

    Args:
        pred: array (H, W) com classes preditas (0 a num_classes-1)
        gt: array (H, W) com classes verdadeiras
        num_classes: numero de classes

    Returns:
        dice_per_class: lista de Dice por classe
        mean_dice: media dos Dice scores
    """
    dice_per_class = None  # TAREFA DO ALUNO: implementar
    mean_dice = None  # TAREFA DO ALUNO: implementar
    return dice_per_class, mean_dice

In [ ]:
# SOLUCAO - Exercicio 1: Dice Score Multi-Classe
def multi_class_dice(pred, gt, num_classes):
    dice_per_class = []
    for cls in range(num_classes):
        pred_mask = (pred == cls)
        gt_mask = (gt == cls)
        intersection = np.logical_and(pred_mask, gt_mask).sum()
        dice = (2 * intersection) / (pred_mask.sum() + gt_mask.sum() + 1e-7)
        dice_per_class.append(dice)
    mean_dice = np.mean(dice_per_class)
    return dice_per_class, mean_dice

# Teste
np.random.seed(42)
H, W = 64, 64
num_classes = 4

# Ground truth: 4 quadrantes = 4 classes
gt = np.zeros((H, W), dtype=int)
gt[:H//2, :W//2] = 0
gt[:H//2, W//2:] = 1
gt[H//2:, :W//2] = 2
gt[H//2:, W//2:] = 3

# Predicao: quase perfeita com algum ruido
pred = gt.copy()
noise_mask = np.random.random((H, W)) < 0.1  # 10% de ruido
pred[noise_mask] = np.random.randint(0, num_classes, size=noise_mask.sum())

dice_per_class, mean_dice = multi_class_dice(pred, gt, num_classes)

print(f'mDice = {mean_dice:.4f}')
print()
for cls in range(num_classes):
    print(f'  Classe {cls}: Dice = {dice_per_class[cls]:.4f}')

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(gt, cmap='tab10', vmin=0, vmax=3)
axes[0].set_title('Ground Truth', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(pred, cmap='tab10', vmin=0, vmax=3)
axes[1].set_title(f'Predicao (10% ruido)', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(pred != gt, cmap='Reds')
axes[2].set_title(f'Erros (mDice={mean_dice:.3f})', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/tmp/multi_class_dice.png', dpi=100, bbox_inches='tight')
plt.show()

### Exercicio 2: Simular Encoder-Decoder
Implemente um encoder-decoder simplificado usando operacoes numpy
(downsampling com mean pooling, upsampling com repeticao) para entender
o efeito da perda de resolucao.

Faca o seguinte:
- Implemente `simple_encoder(img, levels)` que faz downsampling progressivo
- Implemente `simple_decoder(features, original_size)` que faz upsampling
- Compare reconstrucao com e sem "skip connections" (informacao original)

In [ ]:
# PRATICA - Exercicio 2: Encoder-Decoder Simplificado
def simple_encoder(img, levels=3):
    """
    Encoder: downsampling progressivo usando mean pooling.

    Args:
        img: array 2D
        levels: numero de niveis de downsampling

    Returns:
        features: lista de feature maps (do maior ao menor)
    """
    features = None  # TAREFA DO ALUNO: implementar
    return features

def simple_decoder(bottleneck, skip_features=None, target_size=None):
    """
    Decoder: upsampling progressivo.

    Args:
        bottleneck: feature map do nivel mais baixo
        skip_features: lista de features do encoder (para skip connections)
        target_size: tamanho final desejado

    Returns:
        reconstruction: array 2D reconstruido
    """
    reconstruction = None  # TAREFA DO ALUNO: implementar
    return reconstruction

In [ ]:
# SOLUCAO - Exercicio 2: Encoder-Decoder Simplificado
def simple_encoder(img, levels=3):
    features = [img]
    current = img.copy()
    for _ in range(levels):
        h, w = current.shape
        # Mean pooling 2x2
        pooled = current.reshape(h//2, 2, w//2, 2).mean(axis=(1, 3))
        features.append(pooled)
        current = pooled
    return features  # [original, level1, level2, ..., bottleneck]

def simple_decoder(bottleneck, skip_features=None, target_size=None):
    current = bottleneck.copy()
    n_ups = 0
    while current.shape[0] < target_size[0]:
        # Upsample 2x (nearest neighbor)
        h, w = current.shape
        upsampled = np.repeat(np.repeat(current, 2, axis=0), 2, axis=1)

        if skip_features is not None:
            # Skip connection: media entre upsampled e feature do encoder
            skip_idx = len(skip_features) - 2 - n_ups
            if 0 <= skip_idx < len(skip_features):
                skip = skip_features[skip_idx]
                # Ajustar tamanho se necessario
                min_h = min(upsampled.shape[0], skip.shape[0])
                min_w = min(upsampled.shape[1], skip.shape[1])
                upsampled[:min_h, :min_w] = 0.5 * upsampled[:min_h, :min_w] + 0.5 * skip[:min_h, :min_w]

        current = upsampled
        n_ups += 1
    return current[:target_size[0], :target_size[1]]

# Criar imagem de teste com detalhes finos
np.random.seed(42)
img = np.zeros((64, 64))
# Objetos
img[10:20, 10:20] = 1.0  # quadrado
img[30:50, 30:50] = 0.7  # quadrado grande
# Borda fina
img[5, :] = 0.5
img[60, :] = 0.5

# Encoder
features = simple_encoder(img, levels=3)

# Decoder SEM skip connections
recon_no_skip = simple_decoder(features[-1], skip_features=None, target_size=img.shape)

# Decoder COM skip connections
recon_with_skip = simple_decoder(features[-1], skip_features=features, target_size=img.shape)

# Visualizar
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Row 1: encoder stages
for i, feat in enumerate(features):
    ax = axes[0, i]
    ax.imshow(feat, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Level {i} ({feat.shape[0]}x{feat.shape[1]})', fontsize=10, fontweight='bold')
    ax.axis('off')
axes[0, 4].axis('off')
axes[0, 0].set_ylabel('ENCODER', fontsize=12, fontweight='bold')

# Row 2: decoder comparison
axes[1, 0].imshow(img, cmap='viridis', vmin=0, vmax=1)
axes[1, 0].set_title('Original', fontsize=10, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(recon_no_skip, cmap='viridis', vmin=0, vmax=1)
axes[1, 1].set_title('Sem Skip\n(borrado)', fontsize=10, fontweight='bold')
axes[1, 1].axis('off')

axes[1, 2].imshow(recon_with_skip, cmap='viridis', vmin=0, vmax=1)
axes[1, 2].set_title('Com Skip\n(detalhes)', fontsize=10, fontweight='bold')
axes[1, 2].axis('off')

# Error maps
err_no = np.abs(img - recon_no_skip)
err_with = np.abs(img - recon_with_skip)
axes[1, 3].imshow(err_no, cmap='Reds', vmin=0, vmax=0.5)
axes[1, 3].set_title(f'Erro sem skip\n(MAE={err_no.mean():.3f})', fontsize=10, fontweight='bold')
axes[1, 3].axis('off')

axes[1, 4].imshow(err_with, cmap='Reds', vmin=0, vmax=0.5)
axes[1, 4].set_title(f'Erro com skip\n(MAE={err_with.mean():.3f})', fontsize=10, fontweight='bold')
axes[1, 4].axis('off')

plt.suptitle('Encoder-Decoder: Com vs Sem Skip Connections', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/encoder_decoder_demo.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'MAE sem skip connections: {err_no.mean():.4f}')
print(f'MAE com skip connections: {err_with.mean():.4f}')
print(f'Melhoria: {(err_no.mean() - err_with.mean()) / err_no.mean() * 100:.0f}%')

### Exercicio 3: Comparar CE Loss vs Dice Loss
Simule predicoes com diferentes niveis de imbalance e compare o comportamento
de Cross-Entropy e Dice Loss.

Faca o seguinte:
- Para cada fg_ratio em [0.01, 0.05, 0.10, 0.25, 0.50], crie mascaras aleatorias
- Calcule CE e Dice para predicoes corretas e para "predict all background"
- Visualize quando cada loss "engana" o modelo

In [ ]:
# PRATICA - Exercicio 3: CE vs Dice Loss
def pixel_ce_loss(pred_probs, gt, eps=1e-7):
    """
    Cross-Entropy por pixel.
    pred_probs: array (H, W) com probabilidade de foreground
    gt: array (H, W) binario
    """
    loss = None  # TAREFA DO ALUNO: implementar
    return loss

def pixel_dice_loss(pred_probs, gt, smooth=1.0):
    """
    Dice Loss.
    pred_probs: array (H, W) com probabilidade de foreground
    gt: array (H, W) binario
    """
    loss = None  # TAREFA DO ALUNO: implementar
    return loss

In [ ]:
# SOLUCAO - Exercicio 3: CE vs Dice Loss
def pixel_ce_loss(pred_probs, gt, eps=1e-7):
    pred_probs = np.clip(pred_probs, eps, 1 - eps)
    loss = -(gt * np.log(pred_probs) + (1 - gt) * np.log(1 - pred_probs))
    return loss.mean()

def pixel_dice_loss(pred_probs, gt, smooth=1.0):
    intersection = (pred_probs * gt).sum()
    dice = (2 * intersection + smooth) / (pred_probs.sum() + gt.sum() + smooth)
    return 1 - dice

# Comparar para diferentes niveis de imbalance
fg_ratios = [0.01, 0.05, 0.10, 0.25, 0.50]
results = {'fg_ratio': [], 'ce_allbg': [], 'dice_allbg': [], 'ce_perfect': [], 'dice_perfect': []}

np.random.seed(42)
for ratio in fg_ratios:
    H, W = 100, 100
    gt = np.zeros((H, W))
    n_fg = int(H * W * ratio)
    idx = np.random.choice(H * W, n_fg, replace=False)
    gt.flat[idx] = 1.0

    # Predicao: tudo background
    pred_allbg = np.full((H, W), 0.01)
    ce_bg = pixel_ce_loss(pred_allbg, gt)
    dice_bg = pixel_dice_loss(pred_allbg, gt)

    # Predicao: perfeita
    pred_perfect = gt * 0.99 + (1 - gt) * 0.01
    ce_perf = pixel_ce_loss(pred_perfect, gt)
    dice_perf = pixel_dice_loss(pred_perfect, gt)

    results['fg_ratio'].append(ratio)
    results['ce_allbg'].append(ce_bg)
    results['dice_allbg'].append(dice_bg)
    results['ce_perfect'].append(ce_perf)
    results['dice_perfect'].append(dice_perf)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(fg_ratios))
width = 0.35

ax = axes[0]
ax.bar(x - width/2, results['ce_allbg'], width, label='Pred = all BG', color='red', alpha=0.7)
ax.bar(x + width/2, results['ce_perfect'], width, label='Pred = perfeita', color='green', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f'{r*100:.0f}%' for r in fg_ratios])
ax.set_xlabel('% Foreground', fontsize=12)
ax.set_ylabel('CE Loss', fontsize=12)
ax.set_title('Cross-Entropy Loss', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(x - width/2, results['dice_allbg'], width, label='Pred = all BG', color='red', alpha=0.7)
ax.bar(x + width/2, results['dice_perfect'], width, label='Pred = perfeita', color='green', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f'{r*100:.0f}%' for r in fg_ratios])
ax.set_xlabel('% Foreground', fontsize=12)
ax.set_ylabel('Dice Loss', fontsize=12)
ax.set_title('Dice Loss', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('CE vs Dice: Predizer Tudo Background vs Perfeito', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/ce_vs_dice_exercise.png', dpi=100, bbox_inches='tight')
plt.show()

print('Observacao crucial:')
print('  Com 1% FG, CE loss para "all BG" e BAIXO (o modelo nao aprende!)')
print('  Com 1% FG, Dice loss para "all BG" e ALTO (forca o modelo a encontrar o FG)')

### O que observar sobre Post-Processing em Segmentacao

Post-processing pode melhorar mIoU em 2-4%:
1. **Connected Components:** remover ilhas menores que N pixels
2. **Morphological Closing:** fechar buracos pequenos dentro de objetos
3. **CRF (Conditional Random Field):** refinar bordas usando informacao da imagem
4. **Test-Time Augmentation (TTA):** media de predicoes com diferentes augmentations

### O que concluir sobre o Pipeline Completo de Segmentacao

Na pratica, o pipeline de segmentacao tem 5 etapas:
1. Pre-processamento (normalize, resize)
2. Augmentation (rotate, flip, elastic)
3. Modelo (U-Net, DeepLab, SegFormer)
4. Post-processing (morphologia, CRF)
5. Avaliacao (mIoU, mDice, boundary metrics)

Cada etapa contribui para a performance final. Nao basta otimizar o modelo.

### Conexao com outros notebooks sobre Pipeline de ML

O pipeline de segmentacao e analogo ao de classificacao (5A_2) e deteccao (5A_3).
A diferenca principal e que augmentation e avaliacao precisam operar em mascaras
pixel-level, nao em labels ou boxes.

### O que observar sobre Datasets de Segmentacao

| Dataset | Dominio | Classes | Imagens | Tipo |
|---------|---------|---------|---------|------|
| PASCAL VOC | Natural | 21 | 11K | Semantic |
| Cityscapes | Driving | 30 | 5K | Panoptic |
| ADE20K | Indoor/Outdoor | 150 | 25K | Semantic |
| COCO-Stuff | Natural | 172 | 164K | Panoptic |
| ISIC | Dermatologia | 2-8 | 25K | Semantic |

### O que concluir sobre Domain Gap em Segmentacao

Um modelo treinado em Cityscapes (Europa) pode falhar em ruas brasileiras (diferente
sinalizacao, vegetacao, estilo urbano). Domain adaptation e crucial:
- **Fine-tuning:** ajustar com poucos exemplos do novo dominio
- **Style transfer:** transferir aparencia do novo dominio para o antigo
- **Self-training:** gerar pseudo-labels no novo dominio

### Conexao com outros notebooks sobre Generalizacao

Domain gap conecta com 4_1 (regularizacao) e 5A_2 (transfer learning). A questao
fundamental e a mesma: como garantir que o modelo generaliza para dados nao vistos
no treino? Em segmentacao, isso e especialmente critico para aplicacoes medicas.

## 8. Erros Comuns e Armadilhas

### Erro 1: Usar CE Loss com Classes Muito Desbalanceadas
**Problema:** com 99% background, o modelo aprende a predizer tudo como background
e ainda tem loss baixo.
**Solucao:** usar Dice Loss, Focal Loss, ou CE com class weights inversamente
proporcionais a frequencia.

### Erro 2: Ignorar Erros de Borda
**Problema:** mIoU parece bom, mas as mascaras tem bordas imprecisas.
**Solucao:** adicionar Boundary Loss ou avaliar com metricas de borda
(Hausdorff distance, boundary F1).

### Erro 3: Treinar e Testar em Resolucoes Diferentes
**Problema:** treina em 256x256, testa em 512x512. Performance cai.
**Solucao:** treinar em resolucao proxima da inferencia, ou usar multi-scale
testing. Se usar resolucao menor no treino, fazer sliding window no teste.

### Erro 4: Nao Usar Data Augmentation Espacial
**Problema:** modelo overfitta rapidamente em datasets pequenos.
**Solucao:** augmentation agressiva: random crop, rotate, elastic deformation,
color jitter. Para medico: elastic + rotate sao essenciais.

### Erro 5: Reportar Pixel Accuracy em vez de mIoU
**Problema:** pixel accuracy de 95% quando 95% dos pixels sao background.
**Solucao:** SEMPRE usar mIoU ou mDice como metrica principal.

### Erro 6: Esquecer de Transformar Mascaras no Augmentation
**Problema:** augmentar imagem mas nao a mascara correspondente.
**Solucao:** usar bibliotecas que transformam pares (imagem, mascara) atomicamente
(Albumentations com mask_transforms).

### Erro 7: Nao Fazer Post-Processing
**Problema:** mascaras com artefatos (ilhas, buracos).
**Solucao:** connected components analysis + morphological operations
(closing para fechar buracos, opening para remover ilhas).

### O que observar sobre Segmentacao em Tempo Real

Para aplicacoes real-time (> 30 FPS), modelos leves sao necessarios:
- **BiSeNet/BiSeNetV2:** two-stream (spatial + context)
- **STDC:** backbone otimizado para segmentacao
- **ENet:** extremamente leve (0.3M params)
- **Fast-SCNN:** otimizado para mobile

Na pratica, o trade-off e claro: DeepLabv3+ com mIoU=79 a 8 FPS vs BiSeNet com mIoU=73 a 47 FPS.

### O que concluir sobre nnU-Net como Framework Automatico

nnU-Net (2021) automatiza TODA a pipeline de segmentacao medica: escolhe arquitetura,
resolucao, augmentation, loss, pos-processamento. Sem intervencao humana, ganha
a maioria dos desafios de segmentacao medica. Se seu problema e medico, comece com nnU-Net.

### Conexao com outros notebooks sobre Automatizacao de ML

nnU-Net conecta com 4_3 (AutoML) -- ambos automatizam decisoes de design.
A diferenca e que nnU-Net usa heuristicas baseadas em dominio (tamanho do dataset,
resolucao das imagens) em vez de busca generica como NAS.

### Por que em ML: Segmentacao como Ferramenta de Diagnostico

Na medicina, segmentacao precisa pode significar a diferenca entre detectar um tumor
precoce e perder o diagnostico. Aplicacoes incluem: segmentacao de tumores cerebrais,
deteccao de lesoes de pele, analise de retina para diabetes, e contagem de celulas.

### O que observar sobre Segmentacao de Video

Segmentacao de video adiciona dimensao temporal:
- **Frame-by-frame:** aplicar modelo de imagem em cada frame (sem consistencia)
- **Video Object Segmentation (VOS):** propagar mascara do primeiro frame
- **Optical flow:** usar movimento para guiar segmentacao
- **XMem, Cutie:** modelos com memoria para manter consistencia

### O que concluir sobre Foundation Models para Segmentacao

Foundation models estao transformando segmentacao:
- **SAM:** segmenta qualquer objeto com prompt (ponto, box, texto)
- **SEEM:** segmentacao unificada (semantic + instance + panoptic + referencia)
- **Grounded SAM:** DINO (deteccao texto) + SAM (segmentacao) = zero-shot

A era de treinar modelos de segmentacao do zero esta terminando para a maioria das aplicacoes.

### Conexao com outros notebooks sobre Transformers

SegFormer e Mask2Former usam Transformers (5A_5) para segmentacao. A vantagem:
self-attention captura contexto global que CNNs so capturam com muitas camadas.
Isso elimina a necessidade de atrous convolutions e ASPP.

### Por que em ML: Segmentacao em Carros Autonomos

Para um carro autonomo, segmentacao panoptica em tempo real e critica:
- Identificar a estrada (para dirigir)
- Detectar pedestres individuais (para evitar)
- Segmentar sinalizacao (para obedecer)
- Tudo em < 33ms (30 FPS) para resposta segura

### O que observar sobre Weak e Semi-Supervised Segmentation

Anotar mascaras pixel-perfect e caro (1-30 min por imagem). Alternativas:
- **Point supervision:** anotar apenas 1 ponto por objeto (10x mais rapido)
- **Scribble supervision:** rabiscar dentro de cada objeto
- **Bounding box supervision:** usar boxes como supervisao fraca
- **Image-level labels:** apenas dizer quais classes estao presentes

Modelos como MCTformer e AFA conseguem ~85% da performance de fully-supervised
usando apenas image-level labels.

### O que concluir sobre o Custo de Anotacao

O maior bottleneck de segmentacao nao e o modelo, e a ANOTACAO. Uma imagem de
segmentacao medica pode levar 30 minutos para anotar. SAM e foundation models
estao mudando isso: use SAM para gerar anotacoes iniciais, depois refine manualmente.

### Conexao com outros notebooks sobre Transfer Learning

Reduzir necessidade de anotacao conecta com 5A_2 (transfer learning) e
3_4 (semi-supervised learning). O principio e o mesmo: usar conhecimento
pre-existente (modelo pre-treinado, dados nao-anotados) para compensar falta de labels.

### Por que em ML: Segmentacao como Interface Homem-Maquina

Segmentacao interativa (SAM, RITM) permite que usuarios nao-tecnicos "ensinem"
mascaras ao modelo com cliques. Aplicacoes: edicao de fotos, anotacao de dados,
remocao de background, e criacao de conteudo (TikTok, Instagram).

## 9. Resumo e Conexoes

### Hierarquia de Conceitos

```
Segmentacao de Imagens
├── Tipos
│   ├── Semantic (classificar pixels)
│   ├── Instance (distinguir objetos)
│   └── Panoptic (unificar ambos)
├── Arquiteturas
│   ├── Encoder-Decoder (U-Net, SegNet)
│   ├── Atrous/Dilated (DeepLab, PSPNet)
│   ├── Two-Stage (Mask R-CNN)
│   └── Transformer (SegFormer, Mask2Former)
├── Loss Functions
│   ├── Cross-Entropy (baseline)
│   ├── Dice Loss (imbalance)
│   ├── Focal Loss (hard examples)
│   └── Boundary Loss (bordas)
├── Metricas
│   ├── mIoU (padrao)
│   ├── mDice (medico)
│   └── Pixel Accuracy (NAO recomendado)
└── Pratica
    ├── Skip Connections (detalhe espacial)
    ├── Post-Processing (morphologia, CRF)
    └── Foundation Models (SAM, Mask2Former)
```

### Tabela de Conexoes

| Conceito | Conecta com | Relacao |
|----------|-------------|---------|
| Encoder CNN | 5A_1 (CNN fundamentos) | Backbone compartilhado |
| Mask R-CNN | 5A_3 (deteccao) | Estende Faster R-CNN |
| Dice/IoU | 2_4 (metricas) | Metricas de overlap |
| Class imbalance | 5A_2 (Focal Loss) | Mesmo problema, mesma solucao |
| Transformers | 5A_5 (ViT) | SegFormer, Mask2Former |
| Deploy | 6_1 (deploy) | Speed vs accuracy trade-off |

### Checklist de Competencias

- [ ] Sei a diferenca entre semantic, instance, e panoptic segmentation
- [ ] Entendo como U-Net funciona (encoder-decoder + skip connections)
- [ ] Sei o que sao atrous convolutions e por que aumentam receptive field
- [ ] Entendo por que Dice Loss e melhor que CE para dados desbalanceados
- [ ] Sei calcular mIoU e mDice para avaliar segmentacao
- [ ] Entendo o papel de SAM como foundation model para segmentacao

### Proximos Passos
- **5A_5 (Vision Transformers):** SegFormer, DETR, e atencao para visao
- **5B (NLP):** segmentacao de sequencias (NER = "segmentacao" de texto)